# Chapter 7: Tool Calling --- Governed Execution

**Learning objectives:**
1. Wire the GovernedToolGateway triple-gate pattern to the sklearn tool set
2. Understand the three gates: schema validation, policy check, plausibility scoring
3. See fault injection in action (blocked tools, error injection, audit trails)
4. Read and interpret GovernedTranscriptEntry audit logs

In [ ]:
# === Preamble (all notebooks) ===
import os

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}, CUDA: {torch.cuda.is_available()}")

import pandas as pd

In [ ]:
# --- Imports: governed gateway + harness components ---
from knowlytix.harness.governance.governed_gateway import (
    GovernedToolGateway, GatewayResult, GovernedTranscriptEntry,
)
from knowlytix.harness.governance.session import SessionState
from knowlytix.harness.governance.tools import TOOL_DEFINITIONS, execute_tool

from knowlytix.harness.testing.policy import PolicyEngine, PolicyRule
from knowlytix.harness.testing.gateway import FaultProfile

print(f"Tool catalog: {len(TOOL_DEFINITIONS)} tools")
print(f"GatewayResult fields: {list(GatewayResult.__dataclass_fields__.keys())}")
print(f"GovernedTranscriptEntry fields: {list(GovernedTranscriptEntry.__dataclass_fields__.keys())}")

In [ ]:
# --- Build a GMS store and session for the chapter ---
from knowlytix.knowledge.config import DocGMSConfig
from knowlytix.knowledge.store import GMSExpertStore
from knowlytix.core.config import GeometryConfig, TrainConfig

STORE_PATH = "governance_monograph/lab_stores/ch07_gateway"
config = DocGMSConfig(
    store_path=STORE_PATH,
    geometry=GeometryConfig(m=128),
    train=TrainConfig(epochs=150, device=str(DEVICE)),
)
store = GMSExpertStore(config, device=DEVICE)

# Ingest domain knowledge so the store is non-trivial
KNOWLEDGE_PATH = "governed_agent/knowledge/data_science_practices.md"
if os.path.exists(KNOWLEDGE_PATH):
    from knowlytix.knowledge.ingest import ingest_document
    result = ingest_document(store, KNOWLEDGE_PATH, llm=None, config=config, device=DEVICE)
    print(f"Ingested knowledge: {result}")
else:
    print("Knowledge file not found; using empty store")

stats = store.stats()
print(f"Store entities: {stats['entities']}, triples: {stats['triples']}")

In [4]:
# --- Prepare session state: load Adult Income and preprocess ---
state = SessionState()

display, raw = execute_tool("load_dataset", {
    "name": "adult", "builtin": "adult", "test_size": 0.3, "random_state": 42,
}, state)
print(display)

display, raw = execute_tool("preprocess_data", {
    "dataset": "adult", "impute_method": "mean",
    "scale_method": "standard", "encode_method": "ordinal",
}, state)
print(display)
print(f"\nSession datasets: {list(state.datasets.keys())}")
print(f"Session preprocessors: {list(state.preprocessors.keys())}")

Loaded 'adult': 48842 samples, 14 features, target='income', task=classification
  Train: 34189, Test: 14653
Preprocessing done: impute=mean, scale=standard, encode=ordinal
  Features: 14 -> 14
  Numeric cols: 6, Categorical cols: 8

Session datasets: ['adult']
Session preprocessors: ['adult']


## Lab 7.1 --- Wire Up the Governed Gateway

Create a `GovernedToolGateway` with policy rules. Demonstrate compliant calls that pass all three gates and violating calls that are blocked at each gate level.

In [5]:
# --- Lab 7.1a: Create policy rules ---
rules = [
    PolicyRule(
        rule_id="RULE-001",
        action="train_model",
        requirement="preprocessing_done",
        rule_type="requires",
        severity="error",
    ),
    PolicyRule(
        rule_id="RULE-002",
        action="load_dataset",
        requirement="delete_raw_data",
        rule_type="blocks",
        severity="error",
    ),
    PolicyRule(
        rule_id="RULE-003",
        action="explain_shap",
        requirement="model_trained",
        rule_type="requires",
        severity="error",
    ),
]

policy = PolicyEngine(store=store, rules=rules)
print(f"Policy engine created with {len(rules)} rules:")
for r in rules:
    print(f"  {r.rule_id}: {r.action} {r.rule_type} {r.requirement} [{r.severity}]")

Policy engine created with 3 rules:
  RULE-001: train_model requires preprocessing_done [error]
  RULE-002: load_dataset blocks delete_raw_data [error]
  RULE-003: explain_shap requires model_trained [error]


In [6]:
# --- Lab 7.1b: Create the governed gateway ---
gateway = GovernedToolGateway(
    session=state,
    store=store,
    policy=policy,
    theta_plausibility=1.5,
    device=str(DEVICE),
)

print("GovernedToolGateway ready")
print(f"  theta_plausibility = {gateway.theta_plausibility}")
print(f"  tool schemas loaded = {len(gateway._schemas)}")
print(f"  policy rules = {len(gateway.policy._rules)}")
print(f"  transcript length = {len(gateway.transcript)}")

GovernedToolGateway ready
  theta_plausibility = 1.5
  tool schemas loaded = 15
  policy rules = 3
  transcript length = 0


In [7]:
# --- Lab 7.1c: Compliant call --- data_summary on preprocessed Adult ---
# Adult was loaded and preprocessed above.  A data_summary call with
# dataset='adult' is not targeted by any of the three rules, so all four
# gates pass.  theta_plausibility=1.5 is a hand-picked default for this
# lab; Lab 7.3 replaces it with a calibrated cut from the policy store.

result = gateway.execute("data_summary", {"dataset": "adult"}, context="exploration")
print(f"data_summary -> blocked: {result.blocked}")
print(f"  reason: {result.reason}")
print(f"  gate_results: {result.gate_results}")
print(f"  execution_time: {result.execution_time_ms:.1f} ms")
print(f"  display (first 200 chars): {result.display[:200]}")

data_summary -> blocked: False
  reason: All gates passed
  gate_results: {'schema': True, 'policy': True, 'plausibility': True, 'fault_injection': True}
  execution_time: 11.6 ms
  display (first 200 chars): Data summary for 'adult':
                 age workclass        fnlwgt education  education-num      marital-status      occupation relationship   race    sex  capital-gain  capital-loss  hours-per-we


In [8]:
# --- Lab 7.1d: Gate 1 violation --- schema validation failure ---
# Missing required parameter "dataset"
result = gateway.execute("data_summary", {}, context="exploration")
print(f"data_summary({{}}) -> blocked: {result.blocked}")
print(f"  reason: {result.reason}")
print(f"  gate_results: {result.gate_results}")

# Wrong type for parameter
result2 = gateway.execute("data_summary", {"dataset": 123}, context="exploration")
print(f"\ndata_summary(dataset=123) -> blocked: {result2.blocked}")
print(f"  reason: {result2.reason}")
print(f"  gate_results: {result2.gate_results}")

# Unknown tool
result3 = gateway.execute("nonexistent_tool", {"x": 1}, context="exploration")
print(f"\nnonexistent_tool -> blocked: {result3.blocked}")
print(f"  reason: {result3.reason}")
print(f"  gate_results: {result3.gate_results}")

data_summary({}) -> blocked: True
  reason: Schema validation failed: Missing required parameter: dataset
  gate_results: {'schema': False}

data_summary(dataset=123) -> blocked: True
  reason: Schema validation failed: Parameter 'dataset' expected type 'string', got int
  gate_results: {'schema': False}

nonexistent_tool -> blocked: True
  reason: Schema validation failed: Unknown tool: nonexistent_tool
  gate_results: {'schema': False}


In [9]:
# --- Lab 7.1e: Gate 2 violation --- policy engine blocks ---
# RULE-001: train_model requires preprocessing_done
result = gateway.execute("train_model", {
    "dataset": "adult",
    "model_type": "RandomForestClassifier",
    "model_name": "rf_test",
    "params": {"random_state": 42},
}, context="model_training")
print(f"train_model -> blocked: {result.blocked}")
print(f"  reason: {result.reason}")
print(f"  gate_results: {result.gate_results}")
if result.policy_decision:
    print(f"  policy blocking_rules: {result.policy_decision.blocking_rules}")

# RULE-002: load_dataset blocked by delete_raw_data
result2 = gateway.execute("load_dataset", {
    "name": "test_data", "builtin": "wine",
}, context="delete_raw_data")
print(f"\nload_dataset (context=delete_raw_data) -> blocked: {result2.blocked}")
print(f"  reason: {result2.reason}")
print(f"  gate_results: {result2.gate_results}")

# RULE-003: explain_shap requires model_trained
result3 = gateway.execute("explain_shap", {
    "dataset": "adult", "model_name": "rf_test",
}, context="diagnostics")
print(f"\nexplain_shap -> blocked: {result3.blocked}")
print(f"  reason: {result3.reason}")
print(f"  gate_results: {result3.gate_results}")

train_model -> blocked: True
  reason: Policy violation: Rule RULE-001: 'train_model' requires 'preprocessing_done' (not met); GMS graph: 'train_model' requires 'preprocessing_done'; Layer 3 skipped: relation 'allowed_in' not in GMS store
  gate_results: {'schema': True, 'policy': False}
  policy blocking_rules: ['RULE-001', 'gms_graph:train_model->requires->preprocessing_done']

load_dataset (context=delete_raw_data) -> blocked: True
  reason: Policy violation: Rule RULE-002: 'load_dataset' blocked by 'delete_raw_data' in context 'delete_raw_data'; GMS graph: 'load_dataset' blocks 'delete_raw_data'; Layer 3 skipped: relation 'allowed_in' not in GMS store
  gate_results: {'schema': True, 'policy': False}

explain_shap -> blocked: True
  reason: Policy violation: Rule RULE-003: 'explain_shap' requires 'model_trained' (not met); GMS graph: 'explain_shap' requires 'model_trained'; Layer 3 skipped: relation 'allowed_in' not in GMS store
  gate_results: {'schema': True, 'policy': False}


In [10]:
# --- Lab 7.1f: Additional calls --- passes and a surprise block ---
# load_dataset in a safe context (not delete_raw_data)
result = gateway.execute("load_dataset", {
    "name": "wine", "builtin": "wine",
}, context="data_loading")
print(f"load_dataset(wine) -> blocked: {result.blocked}")
print(f"  gate_results: {result.gate_results}")
print(f"  reason: {result.reason[:80]}")

# eda_correlation -- no policy rules target this tool
result2 = gateway.execute("eda_correlation", {
    "dataset": "adult", "method": "pearson",
}, context="exploration")
print(f"\neda_correlation -> blocked: {result2.blocked}")
print(f"  gate_results: {result2.gate_results}")

# preprocess_data -- no policy rules target this tool
result3 = gateway.execute("preprocess_data", {
    "dataset": "wine", "impute_method": "mean",
    "scale_method": "standard",
}, context="preprocessing")
print(f"\npreprocess_data(wine) -> blocked: {result3.blocked}")
print(f"  gate_results: {result3.gate_results}")

load_dataset(wine) -> blocked: True
  gate_results: {'schema': True, 'policy': False}
  reason: Policy violation: Rule RULE-002: 'load_dataset' blocked by 'delete_raw_data' in 

eda_correlation -> blocked: False
  gate_results: {'schema': True, 'policy': True, 'plausibility': True, 'fault_injection': True}

preprocess_data(wine) -> blocked: False
  gate_results: {'schema': True, 'policy': True, 'plausibility': True, 'fault_injection': True}


In [11]:
# --- Lab 7.1g: Inspect the transcript so far ---
print(f"Transcript entries: {len(gateway.transcript)}")
print(f"{'#':<3} {'Tool':<22} {'Blocked':<9} {'Gates':<50} {'Reason (first 60 chars)'}")
print("=" * 135)
for i, entry in enumerate(gateway.transcript, 1):
    gates_str = ", ".join(f"{g}={'PASS' if v else 'FAIL'}" for g, v in entry.gate_results.items())
    print(f"{i:<3} {entry.tool_name:<22} {str(entry.blocked):<9} {gates_str:<50} {entry.reason[:60]}")

# Aggregate summary
summary = gateway.transcript_summary()
print(f"\nSummary: {summary['total_calls']} calls, "
      f"{summary['passed']} passed, {summary['blocked']} blocked "
      f"(block rate {summary['block_rate']:.0%})")
print(f"Gate blocks: {summary['gate_blocks']}")

Transcript entries: 10
#   Tool                   Blocked   Gates                                              Reason (first 60 chars)
1   data_summary           False     schema=PASS, policy=PASS, plausibility=PASS, fault_injection=PASS All gates passed
2   data_summary           True      schema=FAIL                                        Schema validation failed: Missing required parameter: datase
3   data_summary           True      schema=FAIL                                        Schema validation failed: Parameter 'dataset' expected type 
4   nonexistent_tool       True      schema=FAIL                                        Schema validation failed: Unknown tool: nonexistent_tool
5   train_model            True      schema=PASS, policy=FAIL                           Policy violation: Rule RULE-001: 'train_model' requires 'pre
6   load_dataset           True      schema=PASS, policy=FAIL                           Policy violation: Rule RULE-002: 'load_dataset' blocked by '
7   

## Lab 7.2 --- Fault Injection and Recovery

Attach `FaultProfile` objects to the gateway and run diagnostic tools repeatedly. Observe which calls succeed and which are blocked by injected faults. Examine the transcript for the full audit trail.

In [12]:
# --- Lab 7.2a: Create a fresh gateway for the fault injection experiment ---
# Use a clean gateway so the transcript is isolated
import random as _rng
_rng.seed(42)  # reproducible fault injection

state2 = SessionState()

# Load and preprocess Adult in the new session
execute_tool("load_dataset",
             {"name": "adult", "builtin": "adult",
              "test_size": 0.3, "random_state": 42}, state2)
execute_tool("preprocess_data", {"dataset": "adult"}, state2)
# Train a model so diagnostics have something to work with
execute_tool("train_model", {
    "dataset": "adult", "model_type": "RandomForestClassifier",
    "model_name": "rf_fault_test",
    "params": {"n_estimators": 50, "random_state": 42},
}, state2)

# Create gateway without policy rules (focus on fault injection)
gw2 = GovernedToolGateway(
    session=state2,
    store=store,
    policy=PolicyEngine(store=store),  # no rules
    theta_plausibility=1.5,
    device=str(DEVICE),
)

# Add fault profile: 50% error rate on diagnose_* tools
gw2.add_fault_profile(FaultProfile(
    tool_pattern="diagnose_*",
    error_rate=0.5,
    error_message="Service timeout after 30s",
))

print("Fault profile added: diagnose_* with 50% error rate")
print(f"Session state: {len(state2.datasets)} datasets, {len(state2.models)} models")

Fault profile added: diagnose_* with 50% error rate
Session state: 1 datasets, 1 models


In [13]:
# --- Lab 7.2b: Run 10 diagnostic calls and observe fault injection ---
results_fault = []
for i in range(10):
    result = gw2.execute("diagnose_robustness", {
        "dataset": "adult",
        "model_name": "rf_fault_test",
        "noise_levels": [0.05, 0.1, 0.2],
    }, context="diagnostics")
    results_fault.append(result)
    status = "BLOCKED (fault)" if result.fault_injected else (
        "BLOCKED (other)" if result.blocked else "OK"
    )
    print(f"  Call {i+1:>2}: {status:<20} time={result.execution_time_ms:.1f}ms")

successes = sum(1 for r in results_fault if not r.blocked)
faults = sum(1 for r in results_fault if r.fault_injected)
print(f"\nResults: {successes}/10 succeeded, {faults}/10 fault-injected")

  Call  1: OK                   time=253.4ms
  Call  2: BLOCKED (fault)      time=0.1ms
  Call  3: BLOCKED (fault)      time=0.0ms
  Call  4: BLOCKED (fault)      time=0.0ms


  Call  5: OK                   time=253.4ms


  Call  6: OK                   time=254.0ms


  Call  7: OK                   time=253.8ms
  Call  8: BLOCKED (fault)      time=0.1ms
  Call  9: BLOCKED (fault)      time=0.0ms
  Call 10: BLOCKED (fault)      time=0.0ms

Results: 4/10 succeeded, 6/10 fault-injected


In [14]:
# --- Lab 7.2c: Full transcript from the fault experiment ---
print(f"Full transcript ({len(gw2.transcript)} entries):")
print(f"{'#':<3} {'Tool':<24} {'Blocked':<9} {'Fault':<7} {'Time ms':<10} {'Reason (first 50 chars)'}")
print("=" * 110)
for i, entry in enumerate(gw2.transcript, 1):
    print(f"{i:<3} {entry.tool_name:<24} {str(entry.blocked):<9} "
          f"{str(entry.fault_injected):<7} {entry.execution_time_ms:<10.1f} "
          f"{entry.reason[:50]}")

summary2 = gw2.transcript_summary()
print(f"\nAggregate: {summary2}")

Full transcript (10 entries):
#   Tool                     Blocked   Fault   Time ms    Reason (first 50 chars)
1   diagnose_robustness      False     False   253.4      All gates passed
2   diagnose_robustness      True      True    0.1        Fault injected: Service timeout after 30s
3   diagnose_robustness      True      True    0.0        Fault injected: Service timeout after 30s
4   diagnose_robustness      True      True    0.0        Fault injected: Service timeout after 30s
5   diagnose_robustness      False     False   253.4      All gates passed
6   diagnose_robustness      False     False   254.0      All gates passed
7   diagnose_robustness      False     False   253.8      All gates passed
8   diagnose_robustness      True      True    0.1        Fault injected: Service timeout after 30s
9   diagnose_robustness      True      True    0.0        Fault injected: Service timeout after 30s
10  diagnose_robustness      True      True    0.0        Fault injected: Service timeou

In [15]:
# --- Lab 7.2d: Non-diagnostic calls bypass the fault profile ---
# The FaultProfile only matches "diagnose_*", so other tools should always pass
result_safe = gw2.execute("data_summary", {"dataset": "adult"}, context="exploration")
print(f"data_summary -> blocked: {result_safe.blocked}, fault: {result_safe.fault_injected}")

result_safe2 = gw2.execute("leaderboard", {"dataset": "adult"}, context="comparison")
print(f"leaderboard  -> blocked: {result_safe2.blocked}, fault: {result_safe2.fault_injected}")

print(f"\nUpdated transcript: {len(gw2.transcript)} entries total")
updated_summary = gw2.transcript_summary()
print(f"Passed: {updated_summary['passed']}, "
      f"Blocked: {updated_summary['blocked']}, "
      f"Faults: {updated_summary['fault_injections']}")

data_summary -> blocked: False, fault: False
leaderboard  -> blocked: False, fault: False

Updated transcript: 12 entries total
Passed: 6, Blocked: 6, Faults: 6


In [16]:
# --- Lab 7.2e: Visualize gate outcomes as a summary table ---
# Build a DataFrame from the gateway-2 transcript
rows = []
for entry in gw2.transcript:
    row = {
        "tool": entry.tool_name,
        "blocked": entry.blocked,
        "fault_injected": entry.fault_injected,
        "time_ms": round(entry.execution_time_ms, 1),
    }
    row.update({f"gate_{g}": v for g, v in entry.gate_results.items()})
    rows.append(row)

df_transcript = pd.DataFrame(rows)
print("Transcript DataFrame:")
print(df_transcript.to_string(index=False))

print("\nBlock rate by tool:")
print(df_transcript.groupby("tool")["blocked"].mean().to_string())

Transcript DataFrame:
               tool  blocked  fault_injected  time_ms  gate_schema  gate_policy  gate_plausibility  gate_fault_injection
diagnose_robustness    False           False    253.4         True         True               True                  True
diagnose_robustness     True            True      0.1         True         True               True                 False
diagnose_robustness     True            True      0.0         True         True               True                 False
diagnose_robustness     True            True      0.0         True         True               True                 False
diagnose_robustness    False           False    253.4         True         True               True                  True
diagnose_robustness    False           False    254.0         True         True               True                  True
diagnose_robustness    False           False    253.8         True         True               True                  True
diagnose_r

## Lab 7.3 --- Calibrated Plausibility Threshold

The `theta_plausibility=1.5` used in Lab 7.1 was a hand-picked default,
applied against a content store that was not trained on authorization
triples. That configuration renders gate 3 decorative --- the geometric
distance between a context string and a tool name on an untrained
relation has no meaningful operating point. Lab 7.3 upgrades to a
calibrated cut by reusing the *policy* store built in Chapter 4 (Lab 4.3),
whose cap-loss training learned a per-relation admissibility radius
$\rho_r$ for every governance relation in the corpus.

This lab differs from Lab 7.1 in three ways:

1. **Dedicated policy store.** `knowlytix.harness.testing.policy_store.build_policy_store()`
   ingests the data-science governance corpus (24 agents, 20 tools, 241
   seeded triples across 9 relations) and returns a store plus calibrated
   judge. The policy store is separate from the content store because
   its vocabulary is governance-shaped, not data-science-shaped ---
   mixing the two would pollute either calibration. Here we *reload* the
   store already trained in Chapter 4 (`force_rebuild=False`); the full
   500-epoch training only runs once per chapter.
2. **Calibrated theta.** Reading `judge._thresholds["policy_geodesic"]`
   replaces the hand-picked `1.5` with the learned cap radius
   $\rho_{\texttt{uses\_tool}} \approx 1.02$. Under `loss_mode="cap"`
   this value is not a post-hoc logistic fit; it is a parameter the
   training loop optimised jointly with the embedding geometry, against
   the same loss the embeddings were optimised against. The cut and the
   geometry it cuts therefore came out of the same optimisation, which
   is a stronger guarantee than fitting a threshold on top of an
   already-trained store.
3. **Agent-centric probe.** Both `PolicyEngine` (gate 2) and
   `GovernedToolGateway` (gate 3) accept a `plausibility_relation`
   kwarg; setting it to `"uses_tool"` points the geometric layer at the
   relation the store was actually trained on, rather than the generic
   `"should_call"` relation the content store was not trained on.

A subtle consequence: when the `PolicyEngine` is calibrated and given a
`plausibility_relation`, it runs the Layer 3 geometric check *inside* its
own `check()` call. Unauthorized (agent, tool) pairs therefore block at
gate 2 --- not gate 3 --- with the geometric sub-decision already
embedded in the policy verdict. The gateway's own gate 3 then becomes a
redundant second probe on the same relation (or a probe on a different
relation, if you wire it that way). The transcript tells you exactly
which gate fired.

In [ ]:
# --- Lab 7.3a: Load the calibrated policy GMS trained in Chapter 4 ---
# Reuse the data-science governance corpus from Chapter 4 (24 agents,
# 20 tools, 241 seeded triples across 9 relations).  force_rebuild=False
# means we reload the trained store from disk if it exists; the full
# 500-epoch training only runs once.
from knowlytix.harness.testing.policy_store import build_policy_store

policy_store, policy_judge = build_policy_store(
    corpus="data_science",
    stores_dir="gms_agent_builder_ext/lab_stores",
    store_name="ch04_ds_policy",   # reuse Chapter 4's trained artefact
    ingest_epochs=500,
    loss_mode="cap",
    device=DEVICE,
    force_rebuild=False,
    verbose=False,
)

thr = policy_judge._thresholds or {}
theta_calibrated = thr["policy_geodesic"]

print("Calibrated policy-geodesic threshold:")
for key in sorted(thr):
    val = thr[key]
    print(f"  {key:20s} = {val}"
          if isinstance(val, (int, float))
          else f"  {key:20s} = {val}")
print(f"\n  theta_calibrated (policy_geodesic) = {theta_calibrated:.4f}")
print("  vs Lab 7.1 ad-hoc theta_plausibility = 1.5")
print(f"\n  Store: {policy_store.stats()['entities']} entities, "
      f"{policy_store.stats()['relations']} relations, "
      f"{policy_store.stats()['triples']} triples")

In [18]:
# --- Lab 7.3b: PolicyEngine wired to the calibrated judge + uses_tool probe ---
engine_calibrated = PolicyEngine(
    store=policy_store,
    judge=policy_judge,
    plausibility_relation="uses_tool",
    rules=[
        PolicyRule(
            rule_id="R-001",
            action="dangerous_delete",
            requirement="*",
            rule_type="blocks",
            severity="error",
        ),
    ],
)

print("PolicyEngine ready (Layer-1 + Layer-3 calibrated)")
print("  plausibility_relation = uses_tool")
print(f"  policy_geodesic tau   = {theta_calibrated:.4f}")
print(f"  rules                 = {len(engine_calibrated._rules)}")

PolicyEngine ready (Layer-1 + Layer-3 calibrated)
  plausibility_relation = uses_tool
  policy_geodesic tau   = 1.0224
  rules                 = 1


In [19]:
# --- Lab 7.3c: GovernedToolGateway with calibrated theta + uses_tool relation ---
# The sklearn TOOL_DEFINITIONS schemas cover data-loading / modeling /
# explanation tools but not the governance-catalogue tools that the
# data_science policy corpus actually authorizes (deploy_model,
# approve_model, trigger_retrain, diagnose_weakness, ...).  We pass a
# minimal permissive schema list for gate 1 so the probe focuses on
# gate 2 / gate 3 behavior.
governance_tool_schemas = [
    {"name": "train_model",          "parameters": {}},
    {"name": "predict",              "parameters": {}},
    {"name": "deploy_model",         "parameters": {}},
    {"name": "approve_model",        "parameters": {}},
    {"name": "trigger_retrain",      "parameters": {}},
    {"name": "diagnose_weakness",    "parameters": {}},
    {"name": "diagnose_resilience",  "parameters": {}},
    {"name": "diagnose_stability",   "parameters": {}},
    {"name": "diagnose_fairness",    "parameters": {}},
    {"name": "dangerous_delete",     "parameters": {}},
]

gateway_calibrated = GovernedToolGateway(
    session=SessionState(),
    store=policy_store,
    policy=engine_calibrated,
    theta_plausibility=theta_calibrated,
    tool_schemas=governance_tool_schemas,
    device=str(DEVICE),
    plausibility_relation="uses_tool",
)

print("Calibrated gateway ready")
print(f"  theta_plausibility    = {gateway_calibrated.theta_plausibility:.4f} "
      f"(was 1.5 in Lab 7.1)")
print(f"  plausibility_relation = {gateway_calibrated.plausibility_relation}")
print(f"  store                 = policy_store "
      f"({policy_store.stats()['entities']} entities, "
      f"{policy_store.stats()['triples']} triples)")

Calibrated gateway ready
  theta_plausibility    = 1.0224 (was 1.5 in Lab 7.1)
  plausibility_relation = uses_tool
  store                 = policy_store (243 entities, 574 triples)


In [20]:
# --- Lab 7.3d: Eight-probe discrimination test ---
# Four authorized (agent, tool) pairs should ALLOW; four unauthorized
# should BLOCK.  Probes come from Chapter 4's Lab 4.4 so the reader can
# confirm the gateway reaches the same verdicts the policy store itself
# reports.  The `context` parameter doubles as the agent identifier.
probes = [
    # Authorized (authorized by corpus):
    ("weakness_reviewer",    "diagnose_weakness",   "ALLOW"),
    ("resilience_reviewer",  "diagnose_resilience", "ALLOW"),
    ("stability_reviewer",   "diagnose_stability",  "ALLOW"),
    ("modeler_tree",         "train_model",         "ALLOW"),
    # Unauthorized (missing from corpus):
    ("modeler_tree",         "deploy_model",        "BLOCK"),
    ("eda_analyst",          "approve_model",       "BLOCK"),
    ("weakness_reviewer",    "deploy_model",        "BLOCK"),
    ("feature_engineer",     "trigger_retrain",     "BLOCK"),
]

print(f"Probe with calibrated theta = {theta_calibrated:.4f}  "
      f"(relation = uses_tool)\n")
print(f"  {'agent':<22} {'tool':<22} {'plaus':>7}  "
      f"{'blocked_by':<14} {'verdict':<7} expect  ok")
correct = 0
for agent, tool, expected in probes:
    result = gateway_calibrated.execute(tool, args={}, context=agent)
    verdict = "BLOCK" if result.blocked else "ALLOW"
    # The calibrated PolicyEngine embeds Layer-3 inside its check(),
    # so unauthorized pairs block at gate 2.  We still call the
    # gateway's internal scorer for reporting the raw distance score.
    raw_plaus = gateway_calibrated._score_plausibility(agent, tool)
    plaus_s = f"{raw_plaus:.3f}" if raw_plaus is not None else "  n/a"
    gr = result.gate_results
    if gr.get("schema") is False:
        blocked_by = "gate1_schema"
    elif gr.get("policy") is False:
        blocked_by = "gate2_policy"
    elif gr.get("plausibility") is False:
        blocked_by = "gate3_plaus"
    elif gr.get("fault_injection") is False:
        blocked_by = "gate4_fault"
    else:
        blocked_by = "-"
    hit = "ok" if verdict == expected else "XX"
    correct += (verdict == expected)
    print(f"  {agent:<22} {tool:<22} {plaus_s:>7}  "
          f"{blocked_by:<14} {verdict:<7} {expected:<6}  {hit}")

print(f"\n  {correct}/{len(probes)} correct")

Probe with calibrated theta = 1.0224  (relation = uses_tool)

  agent                  tool                     plaus  blocked_by     verdict expect  ok
  weakness_reviewer      diagnose_weakness        0.958  -              ALLOW   ALLOW   ok
  resilience_reviewer    diagnose_resilience      0.898  -              ALLOW   ALLOW   ok


  stability_reviewer     diagnose_stability       0.889  -              ALLOW   ALLOW   ok


  modeler_tree           train_model              0.489  -              ALLOW   ALLOW   ok
  modeler_tree           deploy_model             1.800  gate2_policy   BLOCK   BLOCK   ok
  eda_analyst            approve_model            1.710  gate2_policy   BLOCK   BLOCK   ok


  weakness_reviewer      deploy_model             1.652  gate2_policy   BLOCK   BLOCK   ok
  feature_engineer       trigger_retrain          1.787  gate2_policy   BLOCK   BLOCK   ok

  8/8 correct


### What the transcript tells us

The eight probes land in two clean bands. Authorized pairs score $0.489$ to $0.958$ --- all below the calibrated cap $\rho_{\text{uses\_tool}} = 1.0224$. Unauthorized pairs score $1.652$ to $1.800$ --- all above it. The gap between the highest authorized score ($0.958$) and the lowest unauthorized score ($1.652$) is $0.69$, so the decision is robust to small perturbations of the threshold. No hand-tuning, no heatmap inspection: the cap-loss training (Chapter 4 Lab 4.3) learned the relation-conditioned radius directly from the seeded corpus, and the gateway consumes it as a single scalar.

Note the `blocked_by` column. All four BLOCK verdicts fire at **gate 2 (policy)**, not gate 3. That is because we passed `plausibility_relation="uses_tool"` to `PolicyEngine`, so the engine's own Layer 3 check runs the geometric test internally and returns a policy verdict that already has the geometric decision baked in. The gateway's standalone gate 3 then has nothing left to reject --- the `plaus` column shows the same distance score the engine used, but gate 3 reports pass because the engine already answered the question.

This composition is intentional. Two knobs control where the geometric check lives:

* Set `PolicyEngine(plausibility_relation=...)` --- geometric check happens inside the policy engine. Blocks appear at **gate 2** with the policy reason string. Use this when the policy engine is your primary governance surface.
* Set `GovernedToolGateway(plausibility_relation=...)` *without* `plausibility_relation` on the engine --- geometric check happens at **gate 3**, independent of the policy engine. Use this when you want the gate to probe a *different* relation (e.g. a cross-check on a second store) after the engine has already passed the call.

In Labs 7.1/7.2 neither knob was set, gate 3 queried the untrained `"should_call"` relation against a content store, and plausibility never fired --- which is exactly why the ad-hoc $\theta = 1.5$ looked fine in the transcript even though it was doing no work. Calibration turns the gate from decorative into load-bearing.

## Exercises

### Exercise 7.1 --- Add Five More Policy Rules
Add five additional policy rules covering:
(a) cannot delete data,
(b) must have at least two models before running leaderboard,
(c) fairness check required before any deploy action,
(d) maximum 20 tool calls per session,
(e) explanation required before comparison.
Test that each rule triggers correctly.

### Exercise 7.2 --- Retry-on-Fault Recovery
Implement a retry wrapper around `gateway.execute()`: when a tool call is blocked by a fault, automatically retry once. If the retry also fails, log the permanent failure. Run 20 `diagnose_robustness` calls with the 50% fault profile and compare the effective failure rate with and without retry.

### Exercise 7.3 --- Plausibility Scores for All (Context, Tool) Pairs
Compute `store.score_triple(context, "should_call", tool_name)` for every combination of five contexts (exploration, preprocessing, model_training, diagnostics, deployment) and all 15 tools. Display as a heatmap. Identify which tools are most plausible in each context and adjust `theta_plausibility` so that obviously wrong calls are blocked while correct ones pass.

In [21]:
# --- Exercise 7.1 starter code ---
extra_rules = [
    PolicyRule(rule_id="RULE-004", action="load_dataset",
               requirement="no_deletion", rule_type="blocks",
               context="delete_*", severity="error"),
    PolicyRule(rule_id="RULE-005", action="leaderboard",
               requirement="two_models_trained", rule_type="requires",
               severity="error"),
    PolicyRule(rule_id="RULE-006", action="deploy_*",
               requirement="fairness_check_done", rule_type="requires",
               severity="error"),
    PolicyRule(rule_id="RULE-007", action="*",
               requirement="session_call_limit", rule_type="requires",
               context="over_limit", severity="error"),
    PolicyRule(rule_id="RULE-008", action="leaderboard",
               requirement="explanation_run", rule_type="requires",
               severity="warning"),
]
print("Exercise 7.1: Define and test these rules using gateway.add_policy_rule()")
for r in extra_rules:
    print(f"  {r.rule_id}: {r.action} {r.rule_type} {r.requirement}")

Exercise 7.1: Define and test these rules using gateway.add_policy_rule()
  RULE-004: load_dataset blocks no_deletion
  RULE-005: leaderboard requires two_models_trained
  RULE-006: deploy_* requires fairness_check_done
  RULE-007: * requires session_call_limit
  RULE-008: leaderboard requires explanation_run


In [22]:
# --- Exercise 7.2 starter code ---
def execute_with_retry(gateway, tool_name, args, context, max_retries=1):
    """Execute a tool with automatic retry on fault injection."""
    for attempt in range(1 + max_retries):
        result = gateway.execute(tool_name, args, context)
        if not result.fault_injected:
            return result
        if attempt < max_retries:
            print(f"    Retry {attempt + 1} for {tool_name}...")
    return result  # final attempt, still faulted

print("Exercise 7.2: Run 20 calls with and without retry, compare failure rates")

Exercise 7.2: Run 20 calls with and without retry, compare failure rates


In [23]:
# --- Exercise 7.3 starter code ---
# Goal: make the "untrained relation cannot discriminate" argument
# visible.  Build the same 5x15 plausibility matrix on TWO stores:
#   (a) content store (Labs 7.1/7.2) with should_call -- untrained
#   (b) policy store  (Lab 7.3)      with uses_tool   -- trained on
#                                                       65 (agent, uses_tool, tool)
#                                                       authorization triples
contexts = ["exploration", "preprocessing",
            "model_training", "diagnostics", "deployment"]
tool_names = [t["name"] for t in TOOL_DEFINITIONS]

# (a) Content store, untrained should_call
M_content = [[store.score_triple(c, "should_call", t)
              for t in tool_names] for c in contexts]

# (b) Policy store, trained uses_tool (loaded in Lab 7.3 as policy_store)
M_policy  = [[policy_store.score_triple(c, "uses_tool", t)
              for t in tool_names] for c in contexts]

# Tasks:
# 1. Plot both matrices side by side as heatmaps
# 2. Report (min, max, spread) for each matrix; quantify (b)'s
#    spread relative to (a)'s
# 3. Threshold (b) at theta_calibrated (rho_uses_tool from Lab 7.3)
#    and confirm cells below threshold correspond to authorized
#    (agent, uses_tool, tool) triples in the data_science corpus
# 4. Argue in 2-3 sentences why no theta tuned on (a) could
#    discriminate authorized from unauthorized calls
print(f"Exercise 7.3: build two {len(contexts)}x{len(tool_names)} matrices "
      f"and compare their score spreads")

Exercise 7.3: build two 5x15 matrices and compare their score spreads
